# Chapter 9 &mdash; Exponential Blow-Up in NFA-to-RE Conversion

**Concept 3 of the Chapter 9 decomposition:** *Exponential Blow-Up in NFA-to-RE Conversion*

A graph of size $N$ can have $O(2^N)$ paths, and the RE must name every one of them.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Exponential-Blow-Up-NFA2RE/Concept-Exponential-Blow-Up-NFA2RE.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The produced regular expression can be **exponentially longer** than the NFA. The
reason is structural: a graph with $N$ states can have $O(2^N)$ distinct paths, and an
RE that denotes the language must, in effect, **name them all**.

Each elimination step multiplies label lengths ($m\times n$ bypasses, each
concatenating three labels), so length grows multiplicatively down the sequence.

Two practical consequences:

* the **order** of deletion matters a great deal in practice &mdash; Jove's
  `choose_state_to_del` picks a state with few edges;
* the RE you get is rarely the shortest one; simplification is a separate problem (and
  finding the shortest RE is hard).

## 2. Definitions

### A family of machines whose RE grows fast

In [ ]:
def diamond(k):
    """k diamonds in series -- each doubles the number of paths"""
    lines = ['NFA']
    for i in range(k):
        a, b = 'S%d' % i, 'S%d' % (i+1)
        src = 'I' if i == 0 else a
        dst = 'F' if i == k-1 else b
        lines.append('%s : 0 -> U%d' % (src, i))
        lines.append('%s : 1 -> V%d' % (src, i))
        lines.append('U%d : 0 -> %s' % (i, dst))
        lines.append('V%d : 1 -> %s' % (i, dst))
    return md2mc('\n'.join(lines))

### Convert and measure

In [ ]:
def re_of(N, dellist=None):
    g = mk_gnfa(N)
    _, _, r = del_gnfa_states(g) if dellist is None else del_gnfa_states(g, DelList=dellist)
    return r

<!-- nav-strip -->

---

&larr;&nbsp;[Ch9&nbsp;2.&nbsp;State Elimination: Bypass Edges, Self-Loops, and the $m\times n$ Rule](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-State-Elimination/Concept-State-Elimination.ipynb) &nbsp;&middot;&nbsp; [**Chapter 9** index](https://github.com/ganeshutah/Jove/blob/master/Chapter9/README.md) &nbsp;&middot;&nbsp; [Ch9&nbsp;4.&nbsp;A Non-Trivial Conversion, Step by Step](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Non-Trivial-Conversion/Concept-Non-Trivial-Conversion.ipynb)&nbsp;&rarr;

---

## 3. Tests

Paths double with each diamond; so does the RE length.

In [ ]:
rows = []
for k in range(1, 5):
    N = diamond(k)
    r = re_of(N)
    rows.append((k, len(N["Q"]), 2**k, len(r)))
    print("k=%d : |Q| = %2d, paths = %3d, RE length = %4d" % rows[-1])
assert rows[-1][3] > rows[0][3] * 4, "RE length should grow fast"

Every produced RE is still **correct** &mdash; long, not wrong.

In [ ]:
for k in range(1, 4):
    N = diamond(k)
    r = re_of(N)
    assert iso_dfa(min_dfa(nfa2dfa(N)), min_dfa(nfa2dfa(re2nfa(r))))
    print("k=%d : round-trips to an isomorphic minimal DFA" % k)

**Deletion order matters.** Two orders, two very different lengths.

In [ ]:
N = diamond(3)
inner = sorted(q for q in N["Q"] if q.startswith(('U', 'V')))
outer = sorted(q for q in N["Q"] if not q.startswith(('U', 'V')))
# a DelList must be a permutation of ALL original states
assert sorted(inner + outer) == sorted(N["Q"])
r1 = re_of(N, inner + outer)
r2 = re_of(N, outer + inner)
print("inner-first : RE length %d" % len(r1))
print("outer-first : RE length %d" % len(r2))
assert iso_dfa(min_dfa(nfa2dfa(re2nfa(r1))), min_dfa(nfa2dfa(re2nfa(r2))))
print("same language either way :", True)

The minimal DFA, by contrast, stays small &mdash; the blow-up is in the *notation*.

In [ ]:
for k in range(1, 5):
    N = diamond(k)
    print("k=%d : minimal DFA %2d states, RE %4d characters"
          % (k, len(min_dfa(nfa2dfa(N))["Q"]), len(re_of(N))))

## 4. Exercises


1. Why does a graph with $N$ states have up to $2^N$ paths?
2. Design a deletion order for `diamond(4)` that beats both of the ones above.
3. Is finding the shortest equivalent RE decidable? Is it tractable?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter9/Concept-Exponential-Blow-Up-NFA2RE')